# Eight-hour resume — same 176-deck mix
**Resume your saved model.** Six learning threads, four collectors, and an eight-hour soft limit (or earlier campaign completion). A running batch or evaluation finishes and saves before pausing.

The **same 176 decks—16 per class** remain selected: published anchors, legal variations and broadly randomized legal decks. This failure recovery keeps the original snapshot instead of refreshing it. Saved weights, optimizer, random states, archived opponents and completed lessons carry forward. The completed baseline will not repeat.

Duplicate Storm the Gates plays and crafted-spell physical definitions have new targeted checks. Historical failures remain in diagnostic receipts; the dashboard starts at zero for this run. Checkpoints save every 64 training games.

The first code cell only checks the setup. The next cell is **Start / Resume**—only you run it. Preparation checks scan saved checkpoints and may take about a minute.


In [ ]:
from pathlib import Path
import json, subprocess, sys
from IPython.display import HTML, display
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'tools/train_neural.py').exists())
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from importlib import reload
import hearthstone_sim.neural.dashboard as dashboard_module
Dashboard = reload(dashboard_module).Dashboard
campaign_tool = ROOT/'tools/long_training_campaign.py'
# Preparation, integrity checks and configuration validation only; no games or lessons.
preflight = subprocess.run([sys.executable,str(campaign_tool)],cwd=ROOT,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(preflight.stdout.splitlines()[0] if preflight.returncode == 0 else preflight.stdout)
if preflight.returncode == 0: print('Resume checks passed. Training has not started.')
PREFLIGHT_OK = preflight.returncode == 0
if not PREFLIGHT_OK: raise RuntimeError('Campaign checks failed. Training will not start.')


In [ ]:
# This cell starts only when YOU run it. No training was started while preparing this notebook.
if not globals().get('PREFLIGHT_OK',False): raise RuntimeError('Run the successful checks first.')
import time
screen = None
dashboard = None
last_redraw = 0.0
redraw_events = {'started','iteration_saved','tactical_finished','evaluation_started','evaluation_finished','complete','stopped'}
with subprocess.Popen([sys.executable,str(campaign_tool),'--train'],cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True) as process:
    try:
        for line in process.stdout:
            try: event=json.loads(line)
            except ValueError:
                print(line.rstrip());continue
            if event.get('event')=='campaign_ready':
                print('Continuing saved model at',event['global_game'],'games. Remaining campaign games:',event['remaining_games'])
                if event['complete']:print('Campaign complete. No more training will launch.');continue
                dashboard=Dashboard(event['remaining_games']);screen=display(HTML(dashboard.html()),display_id=True)
                print('Baseline evaluation needed:',event['baseline'])
            elif dashboard is not None:
                dashboard.consume(event)
                now=time.monotonic()
                # Collection events still update internal counters, but redraw only at batch boundaries.
                # Evaluation/lesson heartbeats keep long non-batch phases visibly alive.
                if event.get('event') in redraw_events or (event.get('event') in ('evaluation_game','tactical_progress') and now-last_redraw >= 15):
                    screen.update(HTML(dashboard.html()));last_redraw=now
                if event.get('event') in ('tactical_finished','iteration_saved','evaluation_finished','complete','stopped'):
                    summary={k:event[k] for k in ('event','iteration','global_game','checkpoint','metrics','reason','report','assessment_before','assessment_after') if k in event}
                    print(json.dumps(summary))
        returncode=process.wait()
    except BaseException:
        process.terminate()
        try:process.wait(timeout=10)
        except subprocess.TimeoutExpired:process.kill();process.wait()
        raise
if returncode:raise RuntimeError('Run stopped. Saved checkpoints remain intact; inspect the message above.')
